# Primera Implementación del Modelo SmartStep
Solo toma los 3 porcentajes: zona Alta, zona Media y zona Baja

In [2]:
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Porcentajes.csv")

# Nos quedamos solamente con las columnas necesarias
df = df[[
    "Zona_Alta",
    "Zona_Media",
    "Zona_Baja",
    "Diagnostico"
]]

# Eliminar filas incompletas
df = df.dropna()

# Eliminar espacios sobrantes
df["Diagnostico"] = df["Diagnostico"].str.strip()

#### Separamos los valores de X (los porcentajes) e Y (el diagnostico), y codificamos el diagnotisco

In [3]:
X = df[[
    "Zona_Alta",
    "Zona_Media",
    "Zona_Baja"
]].values

y = df["Diagnostico"].values

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(encoder.classes_)

['Pie Cavo' 'Pie Normal' 'Pie Plano']


#### Separamos los datos en entrenamiento y prueba, y los normalizamos

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#### Creamos la red neuronal

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),

    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),

    tf.keras.layers.Dense(3, activation="softmax")
])

#### La compilamos:

In [6]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

Y entrenamos con los datos de entrenamiento:

In [7]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 129ms/step - accuracy: 0.3714 - loss: 1.0668 - val_accuracy: 0.4444 - val_loss: 1.0477
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5429 - loss: 1.0508 - val_accuracy: 0.4444 - val_loss: 1.0278
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.5143 - loss: 1.0361 - val_accuracy: 0.4444 - val_loss: 1.0110
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4857 - loss: 1.0227 - val_accuracy: 0.5556 - val_loss: 0.9951
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5714 - loss: 1.0091 - val_accuracy: 0.5556 - val_loss: 0.9798
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6286 - loss: 0.9959 - val_accuracy: 0.5556 - val_loss: 0.9648
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6286 - loss: 0.9848 - val_accuracy: 0.5556 - val_loss: 0.9499
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6571 - loss: 0.9726 - val_accuracy: 0.6667 - val_loss

#### Revisamos como nos da el accuracy

In [8]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 1.0000 - loss: 0.2185
Accuracy: 1.0


#### Y por ultimo probamos una nueva pisada

In [9]:
nueva_pisada = [[40, 25, 35]]

nueva_pisada = scaler.transform(nueva_pisada)

prediccion = model.predict(nueva_pisada)

clase = prediccion.argmax(axis=1)[0]

print("Diagnóstico:", encoder.inverse_transform([clase])[0])
for diagnostico, probabilidad in zip(encoder.classes_, prediccion[0]):
    print(f"{diagnostico}: {probabilidad:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
Diagnóstico: Pie Plano
Pie Cavo: 0.09
Pie Normal: 0.35
Pie Plano: 0.56
